# Рекомендательные системы
**Popularity - наиболее популярные товары по кластерам**

In [29]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import hdbscan
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
import openpyxl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.compose import ColumnTransformer
from tqdm import tqdm

In [17]:
#df - транзакционный очищенный датасет + кластеры каждого пользователя
#client_df - агрегированный датасет по пользователям
#df_test - тестовый датасет
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')
df_test = pd.read_parquet('df_test.parquet', engine='fastparquet')

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 695703 entries, 0 to 695702
Data columns (total 39 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   Дата                      695703 non-null  datetime64[ns]
 1   ДатаДоставки              695703 non-null  datetime64[ns]
 2   НомерЗаказаНаСайте        695703 non-null  object        
 3   НовыйСтатус               695703 non-null  category      
 4   СуммаЗаказаНаСайте        695703 non-null  float64       
 5   СуммаДокумента            695703 non-null  float64       
 6   МетодДоставки             695703 non-null  category      
 7   ФормаОплаты               695703 non-null  category      
 8   Регион                    693088 non-null  category      
 9   Группа2                   695703 non-null  category      
 10  Группа3                   695703 non-null  category      
 11  Группа4                   664259 non-null  category      
 12  Ти

In [19]:
print(f"Диапазон дат: {df['ДатаЗаказаНаСайте'].min()} - {df['ДатаЗаказаНаСайте'].max()}")

Диапазон дат: 2017-01-01 00:00:00 - 2018-02-28 00:00:00


In [20]:
client_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80795 entries, 0 to 80794
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Телефон_new        80795 non-null  object 
 1   orders_count       80795 non-null  float64
 2   items_total        80795 non-null  float64
 3   revenue_total      80795 non-null  float64
 4   avg_price          80795 non-null  float64
 5   margin_total       80795 non-null  float64
 6   unique_categories  80795 non-null  int64  
 7   recency_days       80795 non-null  int64  
 8   lifetime_days      80795 non-null  int64  
 9   avg_check          80795 non-null  float64
 10  items_per_order    80795 non-null  float64
 11  cluster_5          80795 non-null  int32  
dtypes: float64(7), int32(1), int64(3), object(1)
memory usage: 7.1+ MB


In [21]:
# Глобальная популярность товаров
max_date = df['ДатаЗаказаНаСайте'].max()
tau = 90

# Считаем глобальную популярность с временным взвешиванием
df_weighted = df.copy()
df_weighted['days_ago'] = (max_date - df_weighted['ДатаЗаказаНаСайте']).dt.days
df_weighted['weight'] = np.exp(-df_weighted['days_ago'] / tau)

global_pop = (
    df_weighted.groupby('ID_SKU')
    .agg(global_weighted_count=('weight', 'sum'))
    .reset_index()
)

print(f"Товаров в глобальной популярности: {len(global_pop)}")
global_pop.head()

Товаров в глобальной популярности: 115165


,ID_SKU,global_weighted_count
0,ID000s000445351,0.020928
1,ID000s000454957,0.042144
2,ID000s000485452,0.364703
3,ID000s000492553,0.291646
4,ID000s000494755,1.544166


In [22]:
# Популярность по кластерам
cluster_pop = (
    df_weighted
    .groupby(['cluster_5', 'ID_SKU'])
    .agg(cluster_weighted_count=('weight', 'sum'))
    .reset_index()
)

# Нормировка на глобальную популярность
cluster_pop = cluster_pop.merge(global_pop, on='ID_SKU', how='left')
cluster_pop['score'] = (
    cluster_pop['cluster_weighted_count'] 
    / np.log1p(cluster_pop['global_weighted_count'])
)

# Сортируем внутри каждого кластера
cluster_pop = cluster_pop.sort_values(['cluster_5', 'score'], ascending=[True, False])

print(f"Строк в cluster_pop: {len(cluster_pop)}")
cluster_pop.head()

Строк в cluster_pop: 191386


,cluster_5,ID_SKU,cluster_weighted_count,global_weighted_count,score
1583,0,ID10005589351,27.068035,441.393326,4.443065
5493,0,IDL00010279048,13.587212,26.561434,4.096954
16375,0,IDL00043615048,14.722277,51.469059,3.717537
5106,0,IDL00006986654,15.412503,79.093424,3.516272
5492,0,IDL00010278856,10.790293,41.848096,2.871545


In [23]:
print(cluster_pop['cluster_5'].value_counts().sort_index())

cluster_5
0    20272
1    82438
2    54675
3    10114
4    23887
Name: count, dtype: int64


In [24]:
# Ячейка 4: История покупок пользователей
user_history = (
    df.groupby('Телефон_new')['ID_SKU']
    .apply(set)
    .reset_index()
)
user_history.columns = ['Телефон_new', 'purchased_items']

print(f"Пользователей с историей: {len(user_history)}")
user_history.head()

Пользователей с историей: 80795


,Телефон_new,purchased_items
0,55555748-48484848484870,{ID9010023489250}
1,55574848-48484848484870,{IDL00044424957}
2,55574848-48484949515179,"{IDL00028435452, IDL00046260654, ID000so-26597..."
3,55574848-48494948544878,"{ID4540452, ID10007476755, IDL00039202856, IDL..."
4,55574848-48495057545270,"{IDL00032269553, IDL00001244351, IDL0004383275..."


In [25]:
# Ячейка 5: Функция формирования рекомендаций
def get_popular_recommendations(user_id, user_cluster, K=10):
    # Получаем историю пользователя
    user_row = user_history[user_history['Телефон_new'] == user_id]
    if len(user_row) == 0:
        purchased = set()
    else:
        purchased = user_row.iloc[0]['purchased_items']
    
    # Топ-товары в кластере пользователя
    cluster_items = cluster_pop[cluster_pop['cluster_5'] == user_cluster]
    
    # Исключаем уже купленные
    candidates = cluster_items[~cluster_items['ID_SKU'].isin(purchased)]
    
    # Отбираем K лучших
    recommendations = candidates.head(K)['ID_SKU'].tolist()
    
    # Если не хватило — дополняем глобально популярными
    if len(recommendations) < K:
        global_items = global_pop.sort_values('global_weighted_count', ascending=False)
        global_items = global_items[~global_items['ID_SKU'].isin(purchased)]
        global_items = global_items[~global_items['ID_SKU'].isin(recommendations)]
        needed = K - len(recommendations)
        recommendations.extend(global_items.head(needed)['ID_SKU'].tolist())
    
    return recommendations[:K]

In [26]:
# Ячейка 6: Оценка на полном датасете
test_users = df_test[['Телефон_new', 'ID_SKU']].copy()
test_users.columns = ['Телефон_new', 'true_item']

# Добавляем кластер пользователя
user_clusters = client_df[['Телефон_new', 'cluster_5']]
test_users = test_users.merge(user_clusters, on='Телефон_new', how='inner')

print(f"Тестовых пользователей: {len(test_users)}")
test_users.head()

Тестовых пользователей: 256001


,Телефон_new,true_item,cluster_5
0,55555748-48484848484870,IDL00003532755,0
1,55574848-48484848484870,IDL00037386654,4
2,55574848-48484949515179,ID10000718250,2
3,55574848-48484949515179,ID5746856,2
4,55574848-48484949515179,ID9010026000048,2


In [35]:
# Группировка теста по пользователям
test_grouped = test_users.groupby('Телефон_new').agg(
    true_items=('true_item', list),
    cluster=('cluster_5', 'first')
).reset_index()

print(f"Пользователей в тесте: {len(test_grouped)}")
print(f"Среднее число товаров в заказе: {test_grouped['true_items'].apply(len).mean():.1f}")

Пользователей в тесте: 80795
Среднее число товаров в заказе: 3.2


In [36]:
#Оценка Popularity (уровень пользователей, K=10)
K = 10
hits = 0
map_sum = 0.0
total = len(test_grouped)

for _, row in tqdm(test_grouped.iterrows(), total=total, desc=f"Popularity K={K}"):
    user_id = row['Телефон_new']
    true_items = row['true_items']
    cluster = row['cluster']
    
    recs = get_popular_recommendations(user_id, cluster, K=K)
    
    hits_in_recs = [item for item in true_items if item in recs]
    
    if len(hits_in_recs) > 0:
        hits += 1
        map_sum += np.mean([1.0 / (recs.index(item) + 1) for item in hits_in_recs])

hr = hits / total
map_score = map_sum / total

print(f"HitRate@{K}: {hr:.4f}")
print(f"MAP@{K}: {map_score:.4f}")

Popularity K=10: 100%|███████████████████| 80795/80795 [07:04<00:00, 190.55it/s]

HitRate@10: 0.0478
MAP@10: 0.0171


In [37]:
# Оценка по кластерам (K=10)
for c in sorted(test_grouped['cluster'].unique()):
    ct = test_grouped[test_grouped['cluster'] == c]
    
    hits_c = 0
    map_c = 0.0
    
    for _, row in ct.iterrows():
        user_id = row['Телефон_new']
        true_items = row['true_items']
        
        recs = get_popular_recommendations(user_id, c, K=10)
        
        hits_in_recs = [item for item in true_items if item in recs]
        if len(hits_in_recs) > 0:
            hits_c += 1
            map_c += np.mean([1.0 / (recs.index(item) + 1) for item in hits_in_recs])
    
    print(f"Кластер {c}: n={len(ct)}, HR@10={hits_c/len(ct):.4f}, MAP@10={map_c/len(ct):.4f}")

Кластер 0: n=17368, HR@10=0.0184, MAP@10=0.0084
Кластер 1: n=12768, HR@10=0.0544, MAP@10=0.0205
Кластер 2: n=19387, HR@10=0.0596, MAP@10=0.0223
Кластер 3: n=5764, HR@10=0.0290, MAP@10=0.0093
Кластер 4: n=25508, HR@10=0.0599, MAP@10=0.0192


In [39]:
# Оценка Popularity для разных K
for K_val in [5, 10, 20]:
    hits_k = 0
    map_k = 0.0
    
    for _, row in test_grouped.iterrows():
        user_id = row['Телефон_new']
        true_items = row['true_items']
        cluster = row['cluster']
        
        recs = get_popular_recommendations(user_id, cluster, K=K_val)
        
        hits_in_recs = [item for item in true_items if item in recs]
        if len(hits_in_recs) > 0:
            hits_k += 1
            map_k += np.mean([1.0 / (recs.index(item) + 1) for item in hits_in_recs])
    
    print(f"K={K_val}: HR = {hits_k/total:.4f}, MAP = {map_k/total:.4f}")

K=5: HR = 0.0306, MAP = 0.0152
K=10: HR = 0.0478, MAP = 0.0171
K=20: HR = 0.0684, MAP = 0.0182
